# Info

* Dataset Generalization / cross-domain
* UAVIDS pipeline'ının VeReMi karşılığı:
  * pseudonym-tabanlı source-rate.
  * Sybil hedef = {16,17,18,19} (pseudonym-çoğaltan); 15 (DataReplaySybil). pseudonym çoğaltmadığı için BİLİNEN manifolda.

# Data

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve

DATA = "/data/mixalldata_clean.csv"
SEEDS = (42,1,2,3,4)

# etiket haritası
NORMAL = 0
SYBIL_TARGET = {16,17,18,19}          # pseudonym-çoğaltan Sybil'ler (zero-day hedef)
# 15 (DataReplaySybil) known manifolda gider (pseudo_fanout=1)
RATE = ["s_pseudo_fanout","s_msgs_per_pseudo","s_pseudo_entropy_norm"]
SRC = "sender"; PSEUDO = "senderPseudo"; LABEL = "class"

# Preprocessing

In [ ]:
# ---------- source-rate (pseudonym-tabanlı) ----------
# UAVIDS'teki source_rate_features'ın VeReMi karşılığı:
#   fan-out  = sender başına farklı pseudonym / mesaj sayısı
#   concentr.= mesaj / farklı pseudonym
#   entropy  = pseudonym dağılımının normalize entropisi
def pseudo_rate_features(df):
    s = df[SRC].astype(str).fillna("NA").values
    p = df[PSEUDO].astype(str).fillna("NA").values
    tmp = pd.DataFrame({"s":s,"p":p}); g = tmp.groupby("s")
    msgcount = g["s"].transform("size").astype(float).values
    fanout   = g["p"].transform("nunique").astype(float).values
    ent_map = {}
    for k,gg in tmp.groupby("s"):
        vc = gg["p"].value_counts().values.astype(float); pr = vc/vc.sum()
        ent_map[k] = float(-(pr*np.log(pr+1e-12)).sum())
    ent = np.array([ent_map[x] for x in s])
    mc = np.clip(msgcount,1,None); fo = np.clip(fanout,1,None)
    return pd.DataFrame({
        "s_pseudo_fanout":     fanout/mc,                     # kimlik-üretme oranı
        "s_msgs_per_pseudo":   msgcount/fo,                   # yoğunlaşma (reciprocal)
        "s_pseudo_entropy_norm": ent/np.log(np.clip(fanout,2,None)),  # Eq(2) Seçenek B guard
    }).fillna(0.0).reset_index(drop=True)

def load():
    df = pd.read_csv(DATA, usecols=[SRC,PSEUDO,LABEL])   # bellek dostu: sadece gerekli kolonlar
    return df.reset_index(drop=True)

In [ ]:
# ---------- LOACO + source-disjoint split (Sybil birleşik hedef) ----------
def loaco_split_veremi(df, seed, val=0.3, test=0.3):
    """Known manifold = benign + tüm Sybil-dışı + DataReplaySybil(15).
       Target = birleşik Sybil {16,17,18,19}. Source-disjoint 'sender' üstünde."""
    rng = np.random.default_rng(seed)
    y = df[LABEL].values
    is_t = np.isin(y, list(SYBIL_TARGET))
    dft = df[is_t]                        # hedef (birleşik Sybil)
    dfk = df[~is_t]                       # known manifold (0..15 dahil)
    idx = rng.permutation(len(dfk)); nv = int(len(idx)*val)
    valk = dfk.iloc[idx[:nv]]; tr = dfk.iloc[idx[nv:]]
    nt = int(len(tr)*test)
    test_ = pd.concat([tr.iloc[:nt], dft])
    tr = tr.iloc[nt:]
    return tr.reset_index(drop=True), valk.reset_index(drop=True), test_.reset_index(drop=True)

def _Z(A, B):
    imp = SimpleImputer(strategy="mean").fit(A)
    sc = StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)), sc.transform(imp.transform(B))

def fit_mahal(X):
    mu = X.mean(0); S = np.cov(X, rowvar=False) + 1e-6*np.eye(X.shape[1])
    return mu, np.linalg.pinv(S)
def mahal(X, mu, P):
    d = X-mu; return np.einsum("ij,jk,ik->i", d, P, d)

# RUN

In [ ]:
# ---------- ANA KOŞU: benign-Mahalanobis (source-rate, source-disjoint) ----------
def run_mahal(seeds=SEEDS):
    df = load()
    feats_all = pseudo_rate_features(df)
    df = pd.concat([df, feats_all], axis=1)
    rocs=[]; seen_fracs=[]; rows=[]
    for seed in seeds:
        tr,valk,test_ = loaco_split_veremi(df, seed)
        ytr = tr[LABEL].values; ye = test_[LABEL].values
        is_t = np.isin(ye, list(SYBIL_TARGET))
        # source-disjoint: hedef sender'ı train'de görülmüşse çıkar
        train_src = set(tr[SRC].astype(str))
        seen = is_t & test_[SRC].astype(str).isin(train_src).values
        sdj = ~seen
        seen_fracs.append(seen[is_t].mean())
        # benign satırlar
        bmask = (ytr==NORMAL)
        A = tr[RATE].reset_index(drop=True); B = test_[RATE].reset_index(drop=True)
        Za,Ze = _Z(A,B)
        mu,P = fit_mahal(Za[bmask]); s = mahal(Ze,mu,P)
        roc = roc_auc_score(is_t[sdj].astype(int), s[sdj]); rocs.append(roc)
        rows.append(sdj.sum())
    print(f"[VeReMi] benign-Mahalanobis source-rate | ROC = {np.mean(rocs):.3f} ± {np.std(rocs):.3f}")
    print(f"  unseen-source Sybil rows/seed: {int(np.mean(rows))} | seen-source frac: {np.mean(seen_fracs)*100:.1f}%")
    return np.mean(rocs), np.std(rocs)
run_mahal()

[VeReMi] benign-Mahalanobis source-rate | ROC = 0.999 ± 0.000
  unseen-source Sybil rows/seed: 977970 | seen-source frac: 0.0%


(np.float64(0.9991753726280967), np.float64(1.3854962424745536e-06))

# Learned direction control

In [ ]:
# ---------- learned-direction kontrol (benign-anchor ≫ known-direction) ----------
from sklearn.linear_model import LogisticRegression
def run_learned(seeds=SEEDS):
    df = load(); df = pd.concat([df, pseudo_rate_features(df)], axis=1)
    rocs=[]
    for seed in seeds:
        tr,valk,test_ = loaco_split_veremi(df, seed)
        ytr = tr[LABEL].values; ye = test_[LABEL].values
        is_t = np.isin(ye, list(SYBIL_TARGET))
        train_src=set(tr[SRC].astype(str))
        seen=is_t & test_[SRC].astype(str).isin(train_src).values; sdj=~seen
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B)
        # known-attack direction: known saldırı (class>0, non-target) vs benign
        yk = ((ytr>0) & (~np.isin(ytr,list(SYBIL_TARGET)))).astype(int)
        clf=LogisticRegression(max_iter=1000).fit(Za, yk)
        s=clf.predict_proba(Ze)[:,1]
        rocs.append(roc_auc_score(is_t[sdj].astype(int), s[sdj]))
    print(f"[VeReMi] learned-direction | ROC = {np.mean(rocs):.3f} ± {np.std(rocs):.3f}  (benign-anchor ile karşılaştır)")
    return np.mean(rocs)
run_learned()

[VeReMi] learned-direction | ROC = 0.680 ± 0.000  (benign-anchor ile karşılaştır)


np.float64(0.6802450663533504)

# Single-feature artifact test

In [ ]:
# ---------- single-feature artefakt testi (trivial mi?) ----------
def artifact_auc(seeds=SEEDS):
    df = load(); df = pd.concat([df, pseudo_rate_features(df)], axis=1)
    per_feat = {f:[] for f in RATE}
    for seed in seeds:
        tr,valk,test_ = loaco_split_veremi(df, seed)
        ye=test_[LABEL].values; is_t=np.isin(ye,list(SYBIL_TARGET))
        train_src=set(tr[SRC].astype(str))
        seen=is_t & test_[SRC].astype(str).isin(train_src).values; sdj=~seen
        for f in RATE:
            v=test_[f].values[sdj]; a=roc_auc_score(is_t[sdj].astype(int), v)
            per_feat[f].append(max(a,1-a))
    print("[VeReMi] single-feature AUC (Sybil vs rest, source-disjoint):")
    for f in RATE: print(f"  {f:24s} {np.mean(per_feat[f]):.3f}")
    print(">>> Biri ~0.99 ise: problem trivial (tek özellik yeter)")
artifact_auc()

[VeReMi] single-feature AUC (Sybil vs rest, source-disjoint):
  s_pseudo_fanout          0.865
  s_msgs_per_pseudo        0.865
  s_pseudo_entropy_norm    0.999
>>> Biri ~0.99 ise: problem trivial (tek özellik yeter) -> dürüstçe 'kolay benchmark' de.


# ROC


In [ ]:
# ---------- Fig: ROC eğrisi noktaları -> CSV (UAVIDS ile aynı format) ----------
def save_roc_csv(seeds=SEEDS, csv="veremi_roc.csv"):
    grid=np.linspace(0,1,101)
    df=load(); df=pd.concat([df,pseudo_rate_features(df)],axis=1)
    curves=[]
    for seed in seeds:
        tr,valk,test_=loaco_split_veremi(df,seed)
        ytr=tr[LABEL].values; ye=test_[LABEL].values; is_t=np.isin(ye,list(SYBIL_TARGET))
        train_src=set(tr[SRC].astype(str))
        seen=is_t & test_[SRC].astype(str).isin(train_src).values; sdj=~seen
        bmask=(ytr==NORMAL)
        A=tr[RATE].reset_index(drop=True); B=test_[RATE].reset_index(drop=True)
        Za,Ze=_Z(A,B); mu,P=fit_mahal(Za[bmask]); s=mahal(Ze,mu,P)
        f,t,_=roc_curve(is_t[sdj].astype(int), s[sdj]); curves.append(np.interp(grid,f,t))
    Arr=np.vstack(curves)
    out=pd.DataFrame([{"detector":"mahal_rate_veremi","fpr":float(x),
                       "tpr_mean":float(m),"tpr_std":float(sd)}
                      for x,m,sd in zip(grid,Arr.mean(0),Arr.std(0))])
    out.to_csv(csv,index=False); print(f"wrote {csv}")
save_roc_csv()

wrote veremi_roc.csv
